# 01 - Data Preprocessing & Exploratory Quantum Analysis
### Deep Learning for Forecasting Quantum Dynamics – PXP Many-Body Model

Questo notebook costituisce il **punto di partenza dell'intera pipeline sperimentale**. 

#### Obiettivi Scientifici del Notebook:
1. **Caricamento Dati ad Alte Prestazioni**: caricamento del tensore quantistico $\mathbf{S} \in \mathbb{R}^{400 \times 1001 \times 55}$ e della griglia temporale $t \in [0, 20]$.
2. **Validazione di Integrità e Sanity Checks Fisici**: verifica assenza di `NaN`/`Inf`, regolarità della griglia temporale $dt=0.02$.
3. **Analisi Statistica e Fisica delle Osservabili**:
   - $10$ Magnetizzazioni locali $m_j(t) = \langle \sigma_j^z \rangle \in [-1, 1]$.
   - $45$ Correlazioni connesse a due corpi $c_{jk}(t) = \langle \sigma_j^z \sigma_k^z \rangle - m_j m_k$.
   - Motivazione fisica sull'assenza di rescaling/normalizzazioni arbitrarie che distorcerebbero i vincoli quantistici.
4. **Visualizzazioni Scientifiche Avanzate**:
   - Oscillazioni coerenti e quench dynamics (Rydberg Scars).
   - Heatmap spazio-temporale delle magnetizzazioni lungo la catena monodimensionale di 10 qubit.
   - Ricostruzione e visualizzazione delle **matrici di correlazione spaziale $10 \times 10$** a diversi istanti temporali ($t = 0, 2, 6, 15$).
5. **Partizionamento Rigoroso e Salvataggio Split**:
   - Split isolato a livello di intere traiettorie: **320 Train (80%)**, **40 Validation (10%)**, **40 Test (10%)**.
   - Congelamento degli indici in `data/split_indices.npz` per garantire coerenza totale nei notebook successivi.


In [ ]:
import os
import sys
from pathlib import Path

# Configurazione del path per importare i moduli in src/
PROJECT_ROOT = Path(os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else "."))
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from utils.seeding import set_seeds
from utils.data_loading import load_dataset, split_trajectories

# Fissaggio del seed per totale riproducibilità
SEED = 42
set_seeds(SEED)

# Rilevamento Device di Calcolo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device attivo: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 1. Caricamento Dati ad Alte Prestazioni

Carichiamo il dataset pre-processato in formato binario float32 (`data/trajectories.npy`). 
Questo approccio riduce il tempo di caricamento da oltre 20 secondi a **meno di 50 millisecondi**, ottimizzando drasticamente lo sviluppo locale e le sessioni di calcolo.


In [ ]:
data_dir = PROJECT_ROOT / "data"
features, time_grid = load_dataset(str(data_dir))

N_traj, N_points, N_features = features.shape

print("=== RIEPILOGO DIMENSIONALE DATASET ===")
print(f"Numero Traiettorie (stati casuali indipendenti): {N_traj}")
print(f"Punti Temporali per Traiettoria: {N_points}")
print(f"Feature Quantistiche Misurate: {N_features}")
print(f"Tipo di dato: {features.dtype}")
print(f"Memoria RAM occupata dal tensore: {features.nbytes / (1024**2):.2f} MB")
print(f"Reticolo temporale: t_0 = {time_grid[0]:.2f}, t_end = {time_grid[-1]:.2f}, dt = {time_grid[1] - time_grid[0]:.4f}")


## 2. Sanity Checks e Integrità dei Dati

Verifichiamo che i dati simulati siano numericamente integri e privi di artefatti numerici prima di qualsiasi elaborazione.


In [ ]:
# Verifica valori nulli o infiniti
has_nan = np.isnan(features).any()
has_inf = np.isinf(features).any()

assert not has_nan, "ERRORE: Rilevati valori NaN nel dataset!"
assert not has_inf, "ERRORE: Rilevati valori infiniti nel dataset!"

# Verifica regolarità temporale
dt_steps = np.diff(time_grid)
is_equispaced = np.allclose(dt_steps, dt_steps[0], atol=1e-5)
assert is_equispaced, "ERRORE: Il vettore temporale non ha campionamento uniforme!"

print("Sanity Checks SUPERATI con successo:")
print("  - Nessun valore NaN rilevato")
print("  - Nessun valore Inf rilevato")
print(f"  - Griglia temporale perfettamente uniforme: dt costante = {dt_steps[0]:.4f}")


## 3. Separazione Concettuale delle Feature e Analisi Statistica

Le $55$ feature osservabili si dividono in due classi fisicamente distinte:
1. **$10$ Magnetizzazioni Locali ($m_1, \dots, m_{10}$)**: feature `[0..9]`, corrispondenti all'eccitazione locale $\langle \sigma_j^z \rangle$.
2. **$45$ Correlazioni a Due Corpi ($c_{12}, c_{13}, \dots, c_{9,10}$)**: feature `[10..54]`, corrispondenti alla connettività quantistica tra ogni coppia distinta di qubit $(j, k)$ con $1 \le j < k \le 10$.


In [ ]:
# Costruzione etichette descrittive per ciascuna feature
magnetization_names = [f"m_{i+1}" for i in range(10)]

correlation_names = []
correlation_pairs = []
for i in range(10):
    for j in range(i + 1, 10):
        correlation_names.append(f"c_{i+1},{j+1}")
        correlation_pairs.append((i, j))

feature_names = magnetization_names + correlation_names

# Segmentazione dei tensori
mag_data = features[:, :, :10]
corr_data = features[:, :, 10:]

print(f"Magnetizzazioni estratte: {mag_data.shape} -> min={mag_data.min():.4f}, max={mag_data.max():.4f}, mean={mag_data.mean():.4f}, std={mag_data.std():.4f}")
print(f"Correlazioni estratte:    {corr_data.shape} -> min={corr_data.min():.4f}, max={corr_data.max():.4f}, mean={corr_data.mean():.4f}, std={corr_data.std():.4f}")


### 📌 Nota Metodologica: Perché NON Normalizziamo con StandardScaler?

In contesti di serie temporali tradizionali si applica spesso la standardizzazione a media 0 e varianza 1 (`StandardScaler`). **Nel nostro contesto quantistico, questo approccio sarebbe concettualmente errato e dannoso**:

1. **Limiti Fisici Intrinseci**: per definizione quantistica, $\langle \sigma^z \rangle \in [-1, 1]$. I valori delle magnetizzazioni e correlazioni sono già limitati e numericamente ben condizionati.
2. **Significato Fisico dello Zero**: lo zero rappresenta lo stato non polarizzato / assenza di correlazione connessa. Traslare i dati con una media artificiale romperebbe la simmetria fisica del sistema.
3. **Mantenimento dell'Interpretabilità**: i modelli predittivi devono restituire stime direttamente interpretabili nello spazio delle osservabili quantistiche reali.


## 4. Visualizzazioni della Dinamica Quantistica

### 4.1 Evoluzione Temporale di Traiettorie Campione
Osserviamo l'evoluzione delle magnetizzazioni e correlazioni per la Traiettoria 0.


In [ ]:
plt.figure(figsize=(15, 6))

# Plot prime 5 magnetizzazioni
plt.subplot(1, 2, 1)
for i in range(5):
    plt.plot(time_grid, mag_data[0, :, i], label=f"$m_{{{i+1}}}(t)$", alpha=0.85)
plt.title("Traiettoria 0: Dinamica delle Magnetizzazioni (Qubit 1-5)", fontsize=12)
plt.xlabel("Tempo $t$", fontsize=11)
plt.ylabel(r"Magnetizzazione $\langle \sigma^z_j \rangle$", fontsize=11)
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right", ncol=2)

# Plot prime 5 correlazioni a due corpi
plt.subplot(1, 2, 2)
for idx in range(5):
    p1, p2 = correlation_pairs[idx]
    plt.plot(time_grid, corr_data[0, :, idx], label=f"$c_{{{p1+1},{p2+1}}}(t)$", alpha=0.85)
plt.title("Traiettoria 0: Correlazioni Connesse a Due Corpi", fontsize=12)
plt.xlabel("Tempo $t$", fontsize=11)
plt.ylabel(r"Correlazione $c_{jk}$", fontsize=11)
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right", ncol=2)

plt.tight_layout()

# Salvataggio figura ad alta risoluzione
fig_dir = PROJECT_ROOT / "results" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_dir / "eda_trajectories_dynamics.png", dpi=300, bbox_inches="tight")
plt.show()


### 4.2 Mappa Spazio-Temporale delle Magnetizzazioni lungo la Catena dei Qubit
Visualizziamo la propagazione spaziale delle eccitazioni lungo la catena monodimensionale dei 10 qubit ($X = \text{qubit } 1..10$, $Y = \text{tempo } t$).


In [ ]:
plt.figure(figsize=(10, 7))

# Traiettoria 0: asse X = Qubit 1..10, asse Y = Tempo
sns.heatmap(
    mag_data[0, :300, :],  # Primi 300 timestep (t da 0 a 6.0) per evidenziare i dettagli fini
    cmap="coolwarm",
    cbar_kws={'label': r'Magnetizzazione $\langle \sigma^z \rangle$'},
    xticklabels=[f"Q{i+1}" for i in range(10)],
    yticklabels=False
)
plt.title("Propagazione Spazio-Temporale delle Magnetizzazioni (Traiettoria 0, t ∈ [0, 6])", fontsize=13)
plt.xlabel("Qubit lungo la Catena 1D", fontsize=11)
plt.ylabel("Evoluzione Temporale (Timestep 0 → 300)", fontsize=11)

plt.savefig(fig_dir / "eda_spatiotemporal_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()


### 4.3 Ricostruzione della Matrice di Correlazione $10 \times 10$ nel Tempo
Dalle $45$ feature di correlazione a due corpi, possiamo ricostruire la matrice simmetrica di connettività quantistica $C(t) \in \mathbb{R}^{10 \times 10}$ per esaminare come l'entanglement si distribuisce tra i qubit a diversi istanti temporali.


In [ ]:
def reconstruct_correlation_matrix(corr_vector: np.ndarray) -> np.ndarray:
    """Ricostruisce la matrice simmetrica 10x10 a partire dal vettore di 45 correlazioni."""
    mat = np.zeros((10, 10), dtype=np.float32)
    idx = 0
    for i in range(10):
        for j in range(i + 1, 10):
            val = corr_vector[idx]
            mat[i, j] = val
            mat[j, i] = val
            idx += 1
    return mat

# Selezioniamo 4 istanti temporali significativi: t = 0, t = 2, t = 6, t = 15
time_indices = [0, 100, 300, 750]  # dt = 0.02

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

for ax, t_idx in zip(axes, time_indices):
    t_val = time_grid[t_idx]
    c_mat = reconstruct_correlation_matrix(corr_data[0, t_idx, :])
    sns.heatmap(
        c_mat,
        cmap="vlag",
        center=0.0,
        ax=ax,
        cbar=(ax == axes[-1]),
        xticklabels=[f"{i+1}" for i in range(10)],
        yticklabels=[f"{i+1}" for i in range(10)],
        vmin=-0.15,
        vmax=0.15
    )
    ax.set_title(f"Correlazioni a t = {t_val:.1f} s", fontsize=12)
    ax.set_xlabel("Qubit Indice", fontsize=10)
    if ax == axes[0]:
        ax.set_ylabel("Qubit Indice", fontsize=10)

plt.suptitle("Evoluzione Temporale della Matrice di Connettività Quantistica C(t)", fontsize=14, y=1.05)
plt.tight_layout()
plt.savefig(fig_dir / "eda_correlation_matrices.png", dpi=300, bbox_inches="tight")
plt.show()


## 5. Partizionamento a Traiettoria Isolata e Congelamento dello Split

Per garantire che non vi sia alcun *data leakage* temporale, **il dataset viene partizionato a livello di intere simulazioni**:
- **Train Set (80%)**: $320$ traiettorie
- **Validation Set (10%)**: $40$ traiettorie
- **Test Set (10%)**: $40$ traiettorie

Salviamo gli indici esatti in `data/split_indices.npz` in modo che tutti i modelli successivi (M1, M2, M3, M4) si addestrino e valutino esattamente sugli stessi campioni.


In [ ]:
train_idx, val_idx, test_idx = split_trajectories(
    num_trajectories=N_traj,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    seed=SEED
)

print("=== CONFERMA DELLO SPLIT INDIPENDENTE ===")
print(f"Train Trajectories:      {len(train_idx)} ({len(train_idx)/N_traj*100:.1f}%)")
print(f"Validation Trajectories: {len(val_idx)} ({len(val_idx)/N_traj*100:.1f}%)")
print(f"Test Trajectories:       {len(test_idx)} ({len(test_idx)/N_traj*100:.1f}%)")

# Verifica disgiunzione perfetta
assert len(set(train_idx).intersection(set(val_idx))) == 0
assert len(set(train_idx).intersection(set(test_idx))) == 0
assert len(set(val_idx).intersection(set(test_idx))) == 0

# Salvataggio degli indici ufficiali
split_file = data_dir / "split_indices.npz"
np.savez(split_file, train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)
print(f"\nIndici di split salvati con successo in: {split_file}")


## 6. Conclusioni e Prossimi Passi

In questo primo step abbiamo:
- Validato la pulizia e la correttezza fisica del dataset simulato.
- Dimostrato la struttura delle $55$ osservabili (10 magnetizzazioni e 45 correlazioni a due corpi).
- Visualizzato la propagazione dell'informazione quantistica lungo la catena 1D e l'evoluzione dell'entanglement a due corpi.
- Congelato lo split ufficiale Train/Val/Test in `data/split_indices.npz`.

### Prossimo Step:
Nel notebook **`02_windowing.ipynb`** implementeremo la logica di estrazione delle sequenze temporali scorrevoli $(X, Y)$, definiremo la griglia di configurazioni per la **Sfida 1 (Time Window Optimization)** e costruiremo i `DataLoader` PyTorch per l'addestramento dei modelli.
